# Module 08 - Multi-Head Attention

This notebook extends Module 07's single-head attention into multi-head attention. The main things to make physical are the reshape, the `sqrt(head_dim)` scaling, matched-parameter head comparisons, and per-head heatmaps.

1. Read the lesson page (`docs/modules/08-multi-head-attention.md`).
2. Open this notebook with `./notebook.sh 08`.
3. Answer the `Question:` / `Answer:` cells below.
4. When you're ready, ask a coding agent to grade your notebook.

Partial work is fine. Blank `Answer: ""` strings are skipped, not counted wrong. If you'd like a hint instead of a grade, write the request inline in the answer string and the agent will tutor first.

In [ ]:
from __future__ import annotations

import math
import subprocess
import sys
from pathlib import Path

import torch

import g2c
from g2c.attention import MultiHeadAttention
from g2c.notebook_extras.multi_head import (
    plot_multi_head_attention_deviation,
    plot_multi_head_attention_heads,
    plot_multi_head_comparison,
    prepare_multi_head_data,
    train_multi_head_comparison,
)

_ = torch.manual_seed(0)
repo_root = Path(g2c.__file__).resolve().parents[1]
experiment_device = "auto"
print("MPS available:", torch.backends.mps.is_available())


## Before the Notebook

Implement `MultiHeadAttention.forward` and `MultiHeadAttention.attention_weights` first. The notebook assumes the Module 08 tests are green.

In [ ]:
"Run from the terminal: .venv/bin/python -m pytest tests/test_multi_head_attention.py -x"
"Question: Which multi-head test is the next one failing, and what implementation detail does it point at?"
"Answer: "

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_multi_head_attention.py"],
    cwd=repo_root,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)

assert result.returncode == 0, "Module 08 multi-head attention tests are not passing yet."
print("Module 08 multi-head attention tests passed.")

## Exercise 1 - Verify Per-Head Scaling by Hand

Force Q and K projections to the identity so each head sees a clean slice of the input. Then compare `MultiHeadAttention.attention_weights` to a manual per-head softmax over `x_h @ x_h.T / sqrt(head_dim)`.

In [ ]:
"Question: With D=4 and H=2, what is head_dim?"
"Answer: "

"Question: Why is the scale sqrt(head_dim), not sqrt(D)?"
"Answer: "

In [ ]:
D, H = 4, 2
head_dim = D // H
mha = MultiHeadAttention(embedding_dim=D, num_heads=H, causal=False)
with torch.no_grad():
    mha.q_proj.W.copy_(torch.eye(D))
    mha.q_proj.b.zero_()
    mha.k_proj.W.copy_(torch.eye(D))
    mha.k_proj.b.zero_()

x = torch.tensor([[[1.0, 0.0, 0.0, 1.0], [0.0, 1.0, 1.0, 0.0]]])
actual = mha.attention_weights(x)
x_per_head = x.view(1, 2, H, head_dim).transpose(1, 2)
expected_scores = x_per_head @ x_per_head.transpose(-2, -1) / math.sqrt(head_dim)
expected = expected_scores.softmax(dim=-1)

print("actual weights shape:", tuple(actual.shape))
print("head 0 weights:")
print(actual[0, 0].detach())
print("head 1 weights:")
print(actual[0, 1].detach())
print("max absolute diff:", (actual - expected).abs().max().item())

assert torch.allclose(actual, expected, atol=1e-5)

## Exercise 2 - Reshape Order Matters

Both reshape orders below produce tensors with valid-looking shapes. They do not assign channels to heads the same way. Trace the numbers and explain which one matches the canonical implementation.

In [ ]:
"Question: What channels should head 0 receive when D=8, H=4, head_dim=2?"
"Answer: "

"Question: Why can a wrong reshape preserve shapes but still change the model?"
"Answer: "

In [ ]:
B, T, D, H = 1, 1, 8, 4
head_dim = D // H
q = torch.arange(D).float().view(B, T, D)

canonical = q.view(B, T, H, head_dim).transpose(1, 2)
swapped = q.view(B, T, head_dim, H).transpose(1, 2)

print("original:", q[0, 0].tolist())
for h in range(H):
    print(f"canonical head {h}:", canonical[0, h, 0].tolist())
print()
for h in range(head_dim):
    print(f"swapped axis {h}:", swapped[0, h, 0].tolist())

## Exercise 3 - Train Tiny LMs at H = 1, 4, 8

This trains small diagnostic causal language models on a 1,000,000-character TinyShakespeare slice for 1,000 AdamW steps. If the saved `ShakespeareTokenizer` artifact exists, the notebook uses it with effective vocab size 1024; otherwise it falls back to a character tokenizer.

These models keep `embedding_dim` fixed, so the attention parameter count is the same. If validation curves differ, the difference comes from the structure of the computation, not from adding more attention parameters.

In [ ]:
"Question: Why is this a fairer comparison than changing D and H together?"
"Answer: "

"Question: What failure mode might appear if head_dim becomes too small?"
"Answer: "

In [ ]:
mha_data = prepare_multi_head_data(repo_root=repo_root)


In [ ]:
head_counts = [1, 4, 8]
comparison = train_multi_head_comparison(
    mha_data,
    head_counts=head_counts,
    device=experiment_device,
)
trained_models = comparison.models
histories = comparison.histories
plot_multi_head_comparison(comparison)


## Exercise 4 - Visualize Per-Head Attention

Pick one trained model and plot one heatmap per head. At this scale the patterns may be partial or noisy. The important thing is that `attention_weights` exposes `(B, H, T, T)`, not an average over heads.

In [ ]:
"Question: Does any head look like a previous-token head? What visual pattern would that be?"
"Answer: "

"Question: Why would averaging over heads hide the point of this visualization?"
"Answer: "

In [ ]:
viz_model = trained_models[4]
weights = plot_multi_head_attention_heads(
    viz_model,
    mha_data,
    prompt="First Citizen: Before we proceed any further, hear me speak.",
    max_tokens=24,
)


### Baseline-Adjusted Attention

The raw heatmap often mostly shows the causal mask. The next plot subtracts the baseline pattern "uniform attention over all allowed previous positions." Red means the head attends more than that baseline; blue means less. If the adjusted plot is nearly white, the head is behaving close to causal-uniform attention.

In [ ]:
centered_weights = plot_multi_head_attention_deviation(
    viz_model,
    mha_data,
    prompt="First Citizen: Before we proceed any further, hear me speak.",
    max_tokens=24,
)


In [ ]:
"Question: Which head deviates most from the causal-uniform baseline?"
"Answer: "

"Question: If all adjusted heatmaps are nearly white, what does that say about this tiny model's head specialization?"
"Answer: "

## Exercise 5 - Parameter Counts at Varying H

The number of heads changes the internal structure but not the total attention parameter count when `D` is fixed.

In [ ]:
"Question: If the parameter count is identical, why can different H values behave differently?"
"Answer: "

In [ ]:
print("D    H    head_dim    expected    actual")
print("-" * 48)
for D, H in [(64, 1), (64, 4), (64, 8), (128, 4), (128, 8)]:
    attn = MultiHeadAttention(embedding_dim=D, num_heads=H)
    expected = 4 * (D * D + D)
    actual = sum(p.numel() for p in attn.parameters())
    print(f"{D:<4d} {H:<4d} {D // H:<10d} {expected:<10d} {actual:<10d}")
    assert expected == actual